# Notebook 02: Discrete HMM with Poisson Jumps (Baseline)

This notebook demonstrates the **discrete** Hidden Markov Model with and without Poisson jump
processes -- the baseline model from the prior paper ([JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl)).
Unlike the continuous Baum-Welch approach developed in our new paper, the discrete model
first **discretizes** observed returns into quantile-based bins and operates entirely on
bin indices. Transition and emission matrices are built from frequency counts on the
discretized sequence.

We compare two variants:
1. **NJ** (No Jumps) -- `MyHiddenMarkovModel`: Markov chain on bin indices with no jump mechanism.
2. **WJ** (With Jumps) -- `MyHiddenMarkovModelWithJumps`: Augments the chain with Poisson-distributed
   regime teleportation events that push the process into tail states.

### Outputs
- **Validation metrics**: KS pass rate, excess kurtosis, ACF-MAE, Wasserstein-1 distance
- **Figure**: Three-panel comparison (density, ACF of |G_t|, Q-Q plot) for NJ vs WJ vs observed
- **Metrics table**: Side-by-side NJ vs WJ summary

## Setup

In [ ]:
include("../Include.jl");

## Configuration

Tune the parameters below to explore different assets, state counts, or jump parameters.

In [ ]:
# --- USER-TUNABLE PARAMETERS ---
ticker = "SPY";             # primary validation asset
K = 13;                     # number of hidden states (= number of emission bins)
n_bins = K;                 # emission bins (kept equal to K for identity emission)
ε = 0.01;                   # jump probability per time step
λ = 3.0;                    # Poisson rate for jump duration
N_PATHS = 1000;             # Monte Carlo simulation paths
L = 252;                    # ACF max lag (1 trading year)
risk_free_rate = 0.0;       # risk-free rate
Δt = 1/252;                 # daily time step in years

## Load Data and Compute Returns

In [ ]:
# --- LOAD IN-SAMPLE DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

# Filter to tickers with complete history
dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

# Compute excess log growth rates for the target ticker
R_obs = log_growth_matrix(dataset, ticker; Δt=Δt, risk_free_rate=risk_free_rate);
println("Loaded $(ticker): T = $(length(R_obs)) observations");

## Discretize Returns into Quantile Bins

We partition the observed continuous returns into `K` quantile-based bins.
Each bin is assigned a **center value** (the mean of returns falling in that bin),
which will be used later to convert simulated bin indices back to continuous returns.

In [ ]:
# --- DISCRETIZE: QUANTILE-BASED BINNING ---
# Compute bin edges from quantiles
quantile_probs = range(0.0, 1.0, length=n_bins+1) |> collect;
bin_edges = quantile(R_obs, quantile_probs);

# Assign each observation to a bin index (1..K)
function assign_bin(x::Float64, edges::Vector{Float64})::Int64
    K_local = length(edges) - 1;
    for k in 1:K_local
        if k == K_local
            return k;  # last bin catches the upper tail
        elseif x < edges[k+1]
            return k;
        end
    end
    return K_local;
end;

bin_indices = [assign_bin(r, bin_edges) for r in R_obs];

# Compute bin centers (mean of returns in each bin)
bin_centers = zeros(K);
for k in 1:K
    mask = findall(x -> x == k, bin_indices);
    if length(mask) > 0
        bin_centers[k] = mean(R_obs[mask]);
    end
end

println("Bin edges: ", round.(bin_edges, digits=1));
println("Bin centers: ", round.(bin_centers, digits=2));
println("Bin counts: ", [count(==(k), bin_indices) for k in 1:K]);

## Build Transition and Emission Matrices

- **Transition matrix** `T[i,j]`: estimated by counting consecutive bin-to-bin transitions and normalizing rows.
- **Emission matrix** `E[i,j]`: set to near-identity (high self-emission probability) since the discrete model emits the bin index corresponding to its current state. A small smoothing floor prevents zero-probability emissions.

In [ ]:
# --- BUILD TRANSITION MATRIX FROM OBSERVED BIN SEQUENCES ---
T_counts = zeros(K, K);
for t in 2:length(bin_indices)
    i = bin_indices[t-1];
    j = bin_indices[t];
    T_counts[i, j] += 1.0;
end

# Normalize rows (add small floor to prevent zero probabilities)
T_mat = copy(T_counts);
for i in 1:K
    row_sum = sum(T_mat[i, :]);
    if row_sum > 0
        T_mat[i, :] ./= row_sum;
    else
        T_mat[i, :] .= 1.0 / K;  # fallback: uniform
    end
end

# --- BUILD EMISSION MATRIX (near-identity with smoothing) ---
emission_floor = 0.001;
E_mat = zeros(K, K);
for i in 1:K
    for j in 1:K
        E_mat[i, j] = (i == j) ? 1.0 : emission_floor;
    end
    E_mat[i, :] ./= sum(E_mat[i, :]);  # normalize
end

println("Transition matrix size: $(size(T_mat))");
println("Row sums (should be 1.0): ", round.(sum(T_mat, dims=2)[:], digits=6));
println("Emission matrix diagonal: ", round.(diag(E_mat), digits=4));

## Build Discrete HMM (No Jumps)

In [ ]:
# --- BUILD DISCRETE HMM (NJ) ---
model_nj = build(MyHiddenMarkovModel, (
    states = collect(1:K),
    T = T_mat,
    E = E_mat
));

println("Discrete HMM (NJ) built: K=$(K) states");

## Build Discrete HMM with Jumps

In [ ]:
# --- BUILD DISCRETE HMM WITH JUMPS (WJ) ---
model_wj = build(MyHiddenMarkovModelWithJumps, (
    states = collect(1:K),
    T = T_mat,
    E = E_mat,
    ϵ = ε,
    λ = λ
));

println("Discrete HMM with Jumps (WJ) built: K=$(K), ε=$(ε), λ=$(λ)");

## Simulate Paths (NJ and WJ)

For each model, we simulate `N_PATHS` state chains starting from the stationary distribution.
Each state chain is converted to continuous returns by mapping `state -> bin_centers[state]`.
The emission step uses `rand(model.emission[state])` to get the emitted bin index, and then
the bin center of that emitted index is used as the continuous return value.

In [ ]:
# --- COMPUTE STATIONARY DISTRIBUTION ---
π_stationary = (T_mat^1000)[1, :];
start_dist = Categorical(π_stationary);
n_steps = length(R_obs);

println("Stationary distribution: ", round.(π_stationary, digits=4));
println("Sum = $(round(sum(π_stationary), digits=6))");

In [ ]:
# --- SIMULATE NJ PATHS ---
sim_nj = Array{Float64,2}(undef, n_steps, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(start_dist);
    state_chain = model_nj(s0, n_steps);
    for t in 1:n_steps
        emitted_bin = rand(model_nj.emission[state_chain[t]]);
        sim_nj[t, i] = bin_centers[emitted_bin];
    end
end

# --- SIMULATE WJ PATHS ---
sim_wj = Array{Float64,2}(undef, n_steps, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(start_dist);
    state_chain = model_wj(s0, n_steps);
    for t in 1:n_steps
        emitted_bin = rand(model_wj.emission[state_chain[t]]);
        sim_wj[t, i] = bin_centers[emitted_bin];
    end
end

println("NJ: $(N_PATHS) paths of length $(n_steps) simulated.");
println("WJ: $(N_PATHS) paths of length $(n_steps) simulated.");

## Validation Metrics

We evaluate each model variant against the observed return distribution using:
- **KS pass rate** -- percentage of paths that pass the two-sample Kolmogorov-Smirnov test at $\alpha = 0.05$
- **Excess kurtosis** -- average simulated kurtosis vs observed
- **ACF-MAE** -- mean absolute error of ACF($|G_t|$) averaged over lags 1 to L
- **Wasserstein-1** -- mean earth-mover distance between sorted quantiles

In [ ]:
# --- VALIDATION METRICS FUNCTION ---
function compute_discrete_metrics(observed::Vector{Float64}, simulated::Matrix{Float64};
                                  α=0.05, L=252)

    n_paths = size(simulated, 2);
    n_obs = length(observed);

    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    acf_obs = autocor(abs.(observed), 1:L);

    # Per-path accumulators
    ks_pass = 0;
    kurt_sum = 0.0; acf_mae_sum = 0.0; w1_sum = 0.0;

    for i in 1:n_paths
        sim = simulated[:, i];

        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α; ks_pass += 1; end

        # Excess kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        if σ_s > 0
            kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;
        end

        # ACF-MAE on absolute returns
        acf_sim = autocor(abs.(sim), 1:L);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));

        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k * length(obs_sorted) / n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k * length(sim_sorted) / n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));
    end

    ks_rate = round(100 * ks_pass / n_paths, digits=1);
    kurt_sim = round(kurt_sum / n_paths, digits=2);
    kurt_o = round(kurt_obs, digits=2);
    acf_mae = round(acf_mae_sum / n_paths, digits=4);
    w1 = round(w1_sum / n_paths, digits=3);

    return (ks_rate=ks_rate, kurtosis_sim=kurt_sim, kurtosis_obs=kurt_o,
            acf_mae=acf_mae, wasserstein=w1)
end;

In [ ]:
# --- COMPUTE METRICS FOR BOTH MODELS ---
metrics_nj = compute_discrete_metrics(R_obs, sim_nj; L=L);
metrics_wj = compute_discrete_metrics(R_obs, sim_wj; L=L);

println("NJ metrics computed.");
println("WJ metrics computed.");

## Figure: NJ vs WJ vs Observed Comparison

Three-panel figure:
- **(a)** Marginal density: observed histogram with NJ and WJ kernel density overlays
- **(b)** ACF($|G_t|$): observed vs NJ mean vs WJ mean with percentile bands
- **(c)** Q-Q plot: observed quantiles vs NJ and WJ quantiles

In [ ]:
# --- PANEL (a): MARGINAL DENSITY ---
p_a = plot(title="(a) Marginal Density",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");

histogram!(p_a, R_obs, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");
density!(p_a, vec(sim_nj), lw=2, color=:blue, alpha=0.7, label="Discrete NJ (K=$K)");
density!(p_a, vec(sim_wj), lw=2, color=:red, alpha=0.7, label="Discrete WJ (K=$K)");

xl_lo = quantile(R_obs, 0.001);
xl_hi = quantile(R_obs, 0.999);
xlims!(p_a, xl_lo, xl_hi);

# --- PANEL (b): ACF(|G|) COMPARISON ---
τ = 1:L;
acf_obs = autocor(abs.(R_obs), τ);

n_acf_paths = min(200, N_PATHS);

# NJ ACF band
acf_nj_archive = hcat([autocor(abs.(sim_nj[:, i]), τ) for i in 1:n_acf_paths]...);
acf_nj_mean = mean(acf_nj_archive, dims=2)[:];
acf_nj_p10 = [quantile(acf_nj_archive[t, :], 0.10) for t in 1:L];
acf_nj_p90 = [quantile(acf_nj_archive[t, :], 0.90) for t in 1:L];

# WJ ACF band
acf_wj_archive = hcat([autocor(abs.(sim_wj[:, i]), τ) for i in 1:n_acf_paths]...);
acf_wj_mean = mean(acf_wj_archive, dims=2)[:];
acf_wj_p10 = [quantile(acf_wj_archive[t, :], 0.10) for t in 1:L];
acf_wj_p90 = [quantile(acf_wj_archive[t, :], 0.90) for t in 1:L];

p_b = plot(τ, acf_obs, lw=2, color=:black, ls=:dash, label="Observed",
    title="(b) ACF(|Gt|) -- Volatility Clustering", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gt|)");
plot!(p_b, τ, acf_nj_mean, lw=2, color=:blue, label="NJ mean");
plot!(p_b, τ, acf_nj_p10, fillrange=acf_nj_p90, alpha=0.12, color=:blue, label="NJ 10-90th");
plot!(p_b, τ, acf_wj_mean, lw=2, color=:red, label="WJ mean");
plot!(p_b, τ, acf_wj_p10, fillrange=acf_wj_p90, alpha=0.12, color=:red, label="WJ 10-90th");

# --- PANEL (c): Q-Q PLOT ---
probs_qq = range(0.001, 0.999, length=200);
q_obs = quantile(R_obs, probs_qq);
q_nj = quantile(vec(sim_nj), probs_qq);
q_wj = quantile(vec(sim_wj), probs_qq);

p_c = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) Q-Q Plot (0.1st-99.9th pctl)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(p_c, q_obs, q_nj, ms=3, alpha=0.6, color=:blue, label="NJ (K=$K)");
scatter!(p_c, q_obs, q_wj, ms=3, alpha=0.6, color=:red, label="WJ (K=$K)");

# --- COMBINE ---
fig = plot(p_a, p_b, p_c, layout=(1, 3), size=(1400, 400),
    plot_title="Discrete HMM Baseline: NJ vs WJ -- $(ticker) (K=$K, ε=$ε, λ=$λ)",
    plot_titlefontsize=12);
display(fig);

savefig(fig, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-Discrete-NJ-vs-WJ-K$(K).svg"));
savefig(fig, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-Discrete-NJ-vs-WJ-K$(K).pdf"));
println("Saved comparison figure to figs/");

## Metrics Table: NJ vs WJ

In [ ]:
# --- PRINT METRICS TABLE ---
println("╔═══════════════════════════════════════════════════════════════════╗");
println("║  Discrete HMM Baseline: $(ticker) (K=$K, N_PATHS=$N_PATHS)              ║");
println("╠═══════════════════════════════════════════════════════════════════╣");
println("║  Metric                 │     NJ      │     WJ      │  Observed  ║");
println("╠═════════════════════════╪═════════════╪═════════════╪════════════╣");
println("║  KS pass rate (α=0.05)  │ $(lpad(metrics_nj.ks_rate, 8))%   │ $(lpad(metrics_wj.ks_rate, 8))%   │     --     ║");
println("║  Excess kurtosis        │ $(lpad(metrics_nj.kurtosis_sim, 9))   │ $(lpad(metrics_wj.kurtosis_sim, 9))   │ $(lpad(metrics_nj.kurtosis_obs, 9)) ║");
println("║  ACF-MAE |Gt|           │ $(lpad(metrics_nj.acf_mae, 9))   │ $(lpad(metrics_wj.acf_mae, 9))   │     --     ║");
println("║  Wasserstein-1          │ $(lpad(metrics_nj.wasserstein, 9))   │ $(lpad(metrics_wj.wasserstein, 9))   │     --     ║");
println("╚═══════════════════════════════════════════════════════════════════╝");

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice, investment recommendations, or trading strategies.